# Preliminaries (Always run)

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install supervision

In [ ]:
import json
import numpy as np
import os
import pandas as pd
import math
import random
import cv2
import torch
import supervision as sv
from PIL import Image
import re

In [ ]:
vitpose_path = "/content/drive/Shareddrives/shoplifting-dataset/vitpose_results_70"
yolo_path = "/content/drive/Shareddrives/shoplifting-dataset/pose_estimation_results_70/YOLOv70"
input_path = "/content/drive/Shareddrives/thesis/dataset/split/Shoplifting"
output_path = "/content/drive/Shareddrives/thesis/dataset"

# Processing Features

In [ ]:
# WIP
# def get_confidences(row):
#   confidences = []
#   for i in range(MAX_DETECTIONS):
#     if row[f"Person_{i}_keypoints"]:
#       for keypoint in row[f"Person_{i}_keypoints"]:
#           confidences.append(keypoint["score"])
#   return np.array(confidences)

# confidence_series = clips_data.map(get_confidences)
# maxes = confidence_series.map(max)
# mins = confidence_series.map(min)
# num_keypoints = sum(confidences_series.map(sum))
# print(min(mins), max(mins))
# print(min(maxes), max(maxes))
# proportions = {}
# for i in range(5, 100, 5):
#   threshold = i/100
#   num_below = sum(confidence_series.map(lambda confs: sum(confs < threshold)))
#   print(threshold, num_clips, num_below/num_keypoints)



In [ ]:
# Features

#Functions currently assume complete required keypoints
joint_indices_all = {'Nose': 0, 'L_Eye': 1, 'R_Eye': 2, 'L_Ear': 3, 'R_Ear': 4, 'L_Shoulder': 5, 'R_Shoulder': 6, 'L_Elbow': 7, 'R_Elbow': 8, 'L_Wrist': 9, 'R_Wrist': 10, 'L_Hip': 11, 'R_Hip': 12, 'L_Knee': 13, 'R_Knee': 14, 'L_Ankle': 15, 'R_Ankle': 16}
# joint_indices_required = {'Neck': 0, 'L_Shoulder': 1, 'R_Shoulder': 2, 'L_Elbow': 3, 'R_Elbow': 4, 'L_Wrist': 5, 'R_Wrist': 6, 'L_Hip': 7, 'R_Hip': 8}
joint_indices_required = {'Neck': 0, 'L_Eye': 1, 'R_Eye': 2, 'L_Shoulder': 3, 'R_Shoulder': 4, 'L_Elbow': 5, 'R_Elbow': 6, 'L_Wrist': 7, 'R_Wrist': 8, 'L_Hip': 9, 'R_Hip': 10}
# Relevant pairs
pairs = [('Neck', 'R_Shoulder'), ('Neck', 'L_Shoulder'), ('R_Shoulder', 'R_Elbow'), ('L_Shoulder', 'L_Elbow'), ('R_Elbow', 'R_Wrist'), ('L_Elbow', 'L_Wrist'), ('R_Wrist', 'R_Hip'), ('L_Wrist', 'L_Hip')]


def add_neck(keypoints):
    l, r = joint_indices_all["L_Shoulder"], joint_indices_all["R_Shoulder"]
    if keypoints[l]["x"] == keypoints[l]["y"] == 0:
      neck_x = keypoints[r]["x"]
      neck_y = keypoints[r]["y"]
    elif keypoints[r]["x"] == keypoints[r]["y"] == 0:
      neck_x = keypoints[l]["x"]
      neck_y = keypoints[l]["y"]
    else:
      neck_x = (keypoints[l]["x"] + keypoints[r]["x"]) / 2
      neck_y = (keypoints[l]["y"] + keypoints[r]["y"]) / 2
    avg_score = (keypoints[l]["score"] + keypoints[r]["score"]) / 2
    return [{"name": "Neck", "x": neck_x, "y": neck_y, "score": avg_score}] + keypoints

def get_required(keypoints):
  '''requires keypoints with neck added'''
  required = []
  for joint in joint_indices_required:
      #gets new index now that neck is included
      ind = joint_indices_all.get(joint, -1) + 1
      required.append(keypoints[ind].copy())
  return required

def normalize_keypoints(keypoints, avg_height):
  normalized = []
  for keypoint in keypoints:
    normalized.append(keypoint.copy())
    normalized[-1]["x"] /= avg_height
    normalized[-1]["y"] /= avg_height
  return normalized

def scale_keypoints(keypoints):
  neck = keypoints[0]
  neck_x = neck["x"]
  neck_y = neck["y"]
  scaled = []
  for keypoint in keypoints:
    scaled.append(keypoint.copy())
    scaled[-1]["x"] -= neck_x
    scaled[-1]["y"] -= neck_y
  return scaled

def get_skeleton(keypoints):
  skeleton = []
  for keypoint in keypoints:
    skeleton.append(keypoint["x"])
    skeleton.append(keypoint["y"])
  return skeleton

def has_required(keypoints):
  '''requires complete unscaled keypoints with neck added'''
  for j in joint_indices_required:
    ind = joint_indices_all.get(j, -1) + 1
    if keypoints[ind]["x"] == keypoints[ind]["y"] == 0:
      return False
  return True

def has_neck(keypoints):
  return keypoints[0]["x"] != 0 or keypoints[0]["y"] != 0

#NOTE: following functions assumes only required keypoints are given

def get_angles(keypoints):
  angles = []
  for joint_i, joint_j in pairs:
    i = joint_indices_required[joint_i]
    j = joint_indices_required[joint_j]
    # if keypoints[i]["x"] == keypoints[i]["y"] == 0 or keypoints[j]["x"] == keypoints[j]["y"] == 0:
    #   angles.append(0)
    # else:
    angles.append(math.atan2(keypoints[i]["y"] - keypoints[j]["y"], keypoints[i]["x"] - keypoints[j]["x"]))
  return angles

def get_lengths(keypoints):
  lengths = []
  for joint_i, joint_j in pairs:
    i = joint_indices_required[joint_i]
    j = joint_indices_required[joint_j]
    # if keypoints[i]["x"] == keypoints[i]["y"] == 0 or keypoints[j]["x"] == keypoints[j]["y"] == 0:
    #   lengths.append(0)
    # else:
    lengths.append(math.sqrt((keypoints[j]["y"] - keypoints[i]["y"]) ** 2 + (keypoints[j]["x"] - keypoints[i]["x"]) ** 2))
  return lengths

def get_motion(keypoints_prev, keypoints_cur):
  motion = []
  for keypoint_prev, keypoint_cur in zip(keypoints_prev, keypoints_cur):
    motion.extend((keypoint_cur["y"] - keypoint_prev["y"], keypoint_cur["x"] - keypoint_prev["x"]))
  return motion

def get_height(keypoints):
  neck = keypoints[joint_indices_required["Neck"]]
  left_hip = keypoints[joint_indices_required["L_Hip"]]
  right_hip = keypoints[joint_indices_required["R_Hip"]]
  x0, y0 = neck["x"], neck["y"]
  x11, y11 = left_hip["x"], left_hip["y"]
  x12, y12 = right_hip["x"], right_hip["y"]
  if not y11 and not y12:
    return 1.0
  if not y11:
    x1, y1 = x12, y12
  elif not y12:
    x1, y1 = x11, y11
  else:
    x1, y1 = (x11 + x12) / 2, (y11 + y12) / 2
  height = math.sqrt((x0 - x1) ** 2 + (y0 - y1) ** 2)
  return height

def get_feature_vector(person_file, good_frames, n_frames=120, padding=None):
  '''padding is one of None, "after", "between"'''
  #TODO: use fpc, work with missing frames
  person = json.load(person_file)
  add_missing(person)
  good_frames = set(good_frames)
  frames = []
  heights = []
  for i in range(n_frames):
    if str(i) not in good_frames:
      frames.append(None)
      continue
    frame = f"frame_{i}"
    #keypoints should have neck
    keypoints = person[frame][0]["keypoints"]
    frames.append(get_required(keypoints))
    heights.append(get_height(frames[-1]))
  avg_height = sum(heights) / len(heights)

  skeletons = []
  angles = []
  motion = []
  lengths = []

  i = 0
  while frames[i] == None:
    if padding == "between":
      skeletons += [0] * 2 * len(joint_indices_required)
      angles += [0] * len(pairs)
      lengths += [0] * (len(pairs))
      motion += [0] * 2 * len(joint_indices_required)
    i += 1

  frame = frames[i]
  normalized_frame = normalize_keypoints(scale_keypoints(frame), avg_height)
  skeletons.extend(get_skeleton(normalized_frame))
  lengths.extend(get_lengths(frame))
  angles.extend(get_angles(frame))
  frame_prev = frame
  i += 1
  while i < n_frames:
    frame = frames[i]
    if frames[i] == None:
      if padding == "between":
        skeletons += [0] * 2 * len(joint_indices_required)
        angles += [0] * len(pairs)
        lengths += [0] * (len(pairs))
        motion += [0] * 2 * len(joint_indices_required)
      i += 1
      continue
    normalized_frame = normalize_keypoints(scale_keypoints(frame), avg_height)
    skeletons.extend(get_skeleton(normalized_frame))
    lengths.extend(get_lengths(frame))
    angles.extend(get_angles(frame))
    motion.extend(get_motion(frame_prev, frame))
    frame_prev = frame
    i += 1

  #pad after
  if padding == "after":
    skeletons += [0] * 2 * len(joint_indices_required) * (n_frames - len(good_frames))
    angles += [0] * len(pairs) * (n_frames - len(good_frames))
    lengths += [0] * (len(pairs)) * (n_frames - len(good_frames))
    motion += [0] * 2 * len(joint_indices_required) * (n_frames - len(good_frames))
  return skeletons + motion + angles + lengths


In [ ]:
def add_missing(person):
  prev = None
  for frame in person:
    if prev == None:
      prev = person[frame][0]["keypoints"]
      continue

    cur = person[frame][0]["keypoints"]
    for i in range(len(cur)):
      keypoint = cur[i]
      if keypoint["x"] == keypoint["y"] == 0:
        keypoint["x"] = prev[i]["x"]
        keypoint["y"] = prev[i]["y"]
        keypoint["score"] = prev[i]["score"]
    prev = cur

In [ ]:
# #updated function
# def add_missing(person):
#   prev_frame = None
#   MIN_FRAMES = 5
#   for frame in person):
#     cur = person[frame][0]["keypoints"]
#     neck = cur[0]
#     if neck["x"] == neck["y"] == 0: #no neck
#       continue

#     if prev_frame == None:
#       prev_frame = frame
#       continue

#     #only update current frame if previous frame is recent
#     dist = int(frame[frame.find('_') + 1:]) - int(prev_frame[prev_frame.find('_') + 1:])
#     if dist <= MIN_FRAMES:
#       prev_frame = frame
#       continue

#     #prev contains the coordinates w.r.t. neck
#     prev = scale_keypoints(person[prev_frame][0]["keypoints"])
#     for i in range(len(cur)):
#       keypoint = cur[i]
#       if keypoint["x"] == keypoint["y"] == 0:
#         keypoint["x"] = neck["x"] + prev[i]["x"]
#         keypoint["y"] = neck["y"] + prev[i]["y"]
#         keypoint["score"] = prev[i]["score"]

#     prev_frame = frame


In [ ]:
#test code
complete_keypoints = [
        {
          "name": "Nose",
          "x": 0.21296,
          "y": 0.197395,
          "score": 0.957988
        },
        {
          "name": "L_Eye",
          "x": 0.21935,
          "y": 0.188178,
          "score": 0.932745
        },
        {
          "name": "R_Eye",
          "x": 0.205316,
          "y": 0.188616,
          "score": 0.927804
        },
        {
          "name": "L_Ear",
          "x": 0.226949,
          "y": 0.19677,
          "score": 0.53724
        },
        {
          "name": "R_Ear",
          "x": 0.192757,
          "y": 0.197992,
          "score": 0.654674
        },
        {
          "name": "L_Shoulder",
          "x": 0.235987,
          "y": 0.257017,
          "score": 0.989575
        },
        {
          "name": "R_Shoulder",
          "x": 0.182495,
          "y": 0.259874,
          "score": 0.945271
        },
        {
          "name": "L_Elbow",
          "x": 0.251025,
          "y": 0.336221,
          "score": 0.924226
        },
        {
          "name": "R_Elbow",
          "x": 1.0,
          "y": 1.0,
          "score": 0.47833
        },
        {
          "name": "L_Wrist",
          "x": 0.249079,
          "y": 0.414618,
          "score": 0.84689
        },
        {
          "name": "R_Wrist",
          "x": 1.0,
          "y": 1.0,
          "score": 0.437973
        },
        {
          "name": "L_Hip",
          "x": 0.224901,
          "y": 0.40113,
          "score": 0.993834
        },
        {
          "name": "R_Hip",
          "x": 0.189735,
          "y": 0.403957,
          "score": 0.984873
        },
        {
          "name": "L_Knee",
          "x": 0.231089,
          "y": 0.507317,
          "score": 0.99508
        },
        {
          "name": "R_Knee",
          "x": 0.195687,
          "y": 0.516484,
          "score": 0.985501
        },
        {
          "name": "L_Ankle",
          "x": 0.234915,
          "y": 0.600612,
          "score": 0.987405
        },
        {
          "name": "R_Ankle",
          "x": 0.206645,
          "y": 0.612397,
          "score": 0.970394
        }
      ]

missing_keypoints = [
  {
    "name": "Nose",
    "x": 0.21296,
    "y": 0.197395,
    "score": 0.957988
  },
  {
    "name": "L_Eye",
    "x": 0.21935,
    "y": 0.188178,
    "score": 0.932745
  },
  {
    "name": "R_Eye",
    "x": 0.205316,
    "y": 0.188616,
    "score": 0.927804
  },
  {
    "name": "L_Ear",
    "x": 0.226949,
    "y": 0.19677,
    "score": 0.53724
  },
  {
    "name": "R_Ear",
    "x": 0.192757,
    "y": 0.197992,
    "score": 0.654674
  },
  {
    "name": "L_Shoulder",
    "x": 0.235987,
    "y": 0.257017,
    "score": 0.989575
  },
  {
    "name": "R_Shoulder",
    "x": 0.0,
    "y": 0.0,
    "score": 0.945271
  },
  {
    "name": "L_Elbow",
    "x": 0.251025,
    "y": 0.336221,
    "score": 0.924226
  },
  {
    "name": "R_Elbow",
    "x": 1.0,
    "y": 1.0,
    "score": 0.47833
  },
  {
    "name": "L_Wrist",
    "x": 0.249079,
    "y": 0.414618,
    "score": 0.84689
  },
  {
    "name": "R_Wrist",
    "x": 1.0,
    "y": 1.0,
    "score": 0.437973
  },
  {
    "name": "L_Hip",
    "x": 0.224901,
    "y": 0.40113,
    "score": 0.993834
  },
  {
    "name": "R_Hip",
    "x": 0.189735,
    "y": 0.403957,
    "score": 0.984873
  },
  {
    "name": "L_Knee",
    "x": 0.231089,
    "y": 0.507317,
    "score": 0.99508
  },
  {
    "name": "R_Knee",
    "x": 0.195687,
    "y": 0.516484,
    "score": 0.985501
  },
  {
    "name": "L_Ankle",
    "x": 0.234915,
    "y": 0.600612,
    "score": 0.987405
  },
  {
    "name": "R_Ankle",
    "x": 0.206645,
    "y": 0.612397,
    "score": 0.970394
  }
]

neck_x = (complete_keypoints[5]["x"] + complete_keypoints[6]["x"]) / 2
neck_y = (complete_keypoints[5]["y"] + complete_keypoints[6]["y"]) / 2
keypoints = add_neck(complete_keypoints)
assert keypoints[0]["x"] == neck_x
assert keypoints[0]["y"] == neck_y
scaled_keypoints = scale_keypoints(keypoints)
for keypoint, scaled in zip(keypoints, scaled_keypoints):
  assert keypoint["x"] - neck_x == scaled["x"]
  assert keypoint["y"] - neck_y == scaled["y"]
clip = {"frame_0": [{"keypoints": add_neck(complete_keypoints)}], "frame_1": [{"keypoints": add_neck(missing_keypoints)}]}
assert not has_required(clip["frame_1"][0]["keypoints"])
add_missing(clip)
assert has_required(clip["frame_1"][0]["keypoints"])
assert clip["frame_1"][0]["keypoints"][2]["x"] == clip["frame_0"][0]["keypoints"][2]["x"]
assert clip["frame_1"][0]["keypoints"][2]["y"] == clip["frame_0"][0]["keypoints"][2]["y"]

In [ ]:
#make uniformly named keypoint json files with neck keypoint added
clips = pd.read_csv(f"{output_path}/shoplifting_ids.csv")
clips["List of Shoplifter ID"] = clips["List of Shoplifter ID"].fillna("[]")
for i in range(len(clips)):
  row = clips.iloc[i]
  clip = row["Clip"]
  print(clip)
  parent = row["Parent Video"]
  label = row["Label"]
  if label == 1:
    ids = json.loads(row["List of Shoplifter ID"])
  elif label == 0:
    ids = json.loads(row["List of Unique ID"])
  else:
    continue
  for id in ids:
    if os.path.exists(f"{yolo_path}/{parent}/{clip}/new_person_{id}.json"):
      with open(f"{yolo_path}/{parent}/{clip}/new_person_{id}.json") as f:
        person = json.load(f)
    else:
      with open(f"{yolo_path}/{parent}/{clip}/person_{id}.json") as f:
        person = json.load(f)

    #add neck to keypoints
    for frame in person:
      person[frame][0]["keypoints"] = add_neck(person[frame][0]["keypoints"])
    os.makedirs(f"{yolo_path}/{parent}/{clip}/persons/person_{id}", exist_ok=True)
    with open(f"{yolo_path}/{parent}/{clip}/persons/person_{id}/keypoints_0.json", 'w') as f:
      json.dump(person, f)

Shoplifting038_x264_24
Shoplifting029_x264_18
Shoplifting001_x264_10
Shoplifting001_x264_11
Shoplifting001_x264_12
Shoplifting001_x264_13
Shoplifting001_x264_14
Shoplifting001_x264_15
Shoplifting001_x264_16
Shoplifting003_x264_56
Shoplifting003_x264_57
Shoplifting003_x264_58
Shoplifting003_x264_59
Shoplifting003_x264_60
Shoplifting003_x264_61
Shoplifting003_x264_62
Shoplifting003_x264_63
Shoplifting004_x264_38
Shoplifting004_x264_39
Shoplifting004_x264_40
Shoplifting004_x264_41
Shoplifting005_x264_6
Shoplifting005_x264_7
Shoplifting006_x264_10
Shoplifting006_x264_11
Shoplifting006_x264_12
Shoplifting006_x264_13
Shoplifting006_x264_14
Shoplifting007_x264_4
Shoplifting007_x264_5
Shoplifting007_x264_6
Shoplifting007_x264_7
Shoplifting007_x264_8
Shoplifting008_x264_42
Shoplifting008_x264_43
Shoplifting008_x264_44
Shoplifting008_x264_45
Shoplifting009_x264_36
Shoplifting009_x264_37
Shoplifting009_x264_38
Shoplifting009_x264_39
Shoplifting009_x264_40
Shoplifting009_x264_41
Shoplifting009_x26

In [ ]:
#code to create ids_csv for training
ids_df = pd.DataFrame(columns=["Clip", "Parent Video", "ID", "Label", "Has Required"])
clips = pd.read_csv(f"{output_path}/shoplifting_ids.csv")
clips["List of Shoplifter ID"] = clips["List of Shoplifter ID"].fillna("[]")
for i in range(len(clips)):
  row = clips.iloc[i]
  new_row = {}
  clip = row["Clip"]
  print(clip)
  parent = row["Parent Video"]
  label = row["Label"]
  if label == 1:
    ids = json.loads(row["List of Shoplifter ID"])
  elif label == 0:
    ids = json.loads(row["List of Unique ID"])
  else:
    continue
  for id in ids:
    new_row["Clip"] = clip
    new_row["Parent Video"] = parent
    new_row["ID"] = id
    new_row["Label"] = label

    ids_df.loc[len(ids_df)] = new_row

ids_df.to_csv(f"{output_path}/ids.csv", index=False)

Shoplifting038_x264_24
Shoplifting029_x264_18
Shoplifting001_x264_10
Shoplifting001_x264_11
Shoplifting001_x264_12
Shoplifting001_x264_13
Shoplifting001_x264_14
Shoplifting001_x264_15
Shoplifting001_x264_16
Shoplifting003_x264_56
Shoplifting003_x264_57
Shoplifting003_x264_58
Shoplifting003_x264_59
Shoplifting003_x264_60
Shoplifting003_x264_61
Shoplifting003_x264_62
Shoplifting003_x264_63
Shoplifting004_x264_38
Shoplifting004_x264_39
Shoplifting004_x264_40
Shoplifting004_x264_41
Shoplifting005_x264_6
Shoplifting005_x264_7
Shoplifting006_x264_10
Shoplifting006_x264_11
Shoplifting006_x264_12
Shoplifting006_x264_13
Shoplifting006_x264_14
Shoplifting007_x264_4
Shoplifting007_x264_5
Shoplifting007_x264_6
Shoplifting007_x264_7
Shoplifting007_x264_8
Shoplifting008_x264_42
Shoplifting008_x264_43
Shoplifting008_x264_44
Shoplifting008_x264_45
Shoplifting009_x264_36
Shoplifting009_x264_37
Shoplifting009_x264_38
Shoplifting009_x264_39
Shoplifting009_x264_40
Shoplifting009_x264_41
Shoplifting009_x26

In [ ]:
ids_df = pd.read_csv(f"{output_path}/ids.csv")
counts = ids_df["Has Required"].map(lambda s: len(json.loads(s)))
print(len(ids_df))
ids_df = ids_df[counts >= 50]
print(len(ids_df))


5558
1200


In [ ]:
n_frames = 120
total_features = 2 * len(joint_indices_required) * n_frames + 2 * len(joint_indices_required) * (n_frames - 1) + len(pairs) * n_frames + len(pairs) * n_frames
print(total_features)
for i in range(len(ids_df)):
  row = ids_df.iloc[i]
  person_path = f"{yolo_path}/{row['Parent Video']}/{row['Clip']}/scaled_person_{row['ID']}.json"
  good_frames = json.loads(row["Has Required"])
  with open(person_path) as f:
    padded_after = get_feature_vector(f, good_frames, padding="after")
    assert len(padded_after) == total_features, f"Vector is of length {len(padded_after)}, expected {total_features}"
  with open(person_path) as f:
    padded_between = get_feature_vector(f, good_frames, padding="between")
    assert len(padded_between) == total_features, f"Vector is of length {len(padded_between)}, expected {total_features}"

7178


In [ ]:
def feature_vector_from_row(row):
  person_path = f"{yolo_path}/{row['Parent Video']}/{row['Clip']}/persons/person_{row['ID']}/keypoints_0.json"
  good_frames = row["Has Required"]
  with open(person_path) as f:
    vector = get_feature_vector(f, good_frames, padding="after")
  return vector

def get_good_frames(row):
  print(f"processing {row['Clip']}; person id: {row['ID']}")
  person_path = f"{yolo_path}/{row['Parent Video']}/{row['Clip']}/persons/person_{row['ID']}/keypoints_0.json"
  with open(person_path) as f:
    person = json.load(f)
  add_missing(person)
  required = []
  for frame in person:
    keypoints = person[frame][0]["keypoints"]
    #if has_required(keypoints):
    if has_neck(keypoints):
      required.append(frame[frame.find('_')+1:])
  return required


ids_df = pd.read_csv(f"{output_path}/ids.csv")
ids_df["Has Required"] = ids_df.apply(get_good_frames, axis=1)
counts = ids_df["Has Required"].map(len)
ids_df = ids_df[counts > 0]
feature_df = pd.DataFrame()
feature_df["clip"] = ids_df["Clip"]
feature_df["parent"] = ids_df["Parent Video"]
feature_df["id"] = ids_df["ID"]
feature_df["features"] = ids_df.apply(feature_vector_from_row, axis=1)
feature_df["target"] = ids_df["Label"]
print(len(feature_df), feature_df["target"].value_counts())
feature_df.to_csv(f"{output_path}/features_eyes.csv", index=False)
feature_df.head()

Streaming output truncated to the last 5000 lines.
processing Shoplifting003_x264_82; person id: 2
processing Shoplifting003_x264_83; person id: 1
processing Shoplifting003_x264_83; person id: 2
processing Shoplifting003_x264_84; person id: 1
processing Shoplifting003_x264_86; person id: 1
processing Shoplifting003_x264_86; person id: 2
processing Shoplifting003_x264_87; person id: 1
processing Shoplifting003_x264_87; person id: 2
processing Shoplifting003_x264_87; person id: 3
processing Shoplifting003_x264_88; person id: 1
processing Shoplifting003_x264_89; person id: 1
processing Shoplifting004_x264_0; person id: 1
processing Shoplifting004_x264_0; person id: 2
processing Shoplifting004_x264_0; person id: 3
processing Shoplifting004_x264_1; person id: 1
processing Shoplifting004_x264_1; person id: 2
processing Shoplifting004_x264_2; person id: 1
processing Shoplifting004_x264_2; person id: 2
processing Shoplifting004_x264_2; person id: 3
processing Shoplifting004_x264_3; person id: 

,clip,parent,id,features,target
0,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,"[0.0, 0.0, -3.6674314484295922, -3.86284365242...",1.0
1,Shoplifting001_x264_10,Shoplifting001_x264.mp4,2,"[0.0, 0.0, -4.92564021544842, -3.3099893501284...",1.0
2,Shoplifting001_x264_11,Shoplifting001_x264.mp4,1,"[0.0, 0.0, -5.068451145658348, -3.374883022677...",1.0
3,Shoplifting001_x264_11,Shoplifting001_x264.mp4,5,"[0.0, 0.0, -1.5864730109297471, -1.27144951956...",1.0
4,Shoplifting001_x264_12,Shoplifting001_x264.mp4,1,"[0.0, 0.0, -5.148482308764561, -3.334219974371...",1.0


In [ ]:
df = pd.read_csv(f"{output_path}/features_eyes.csv")
df["features"] = df["features"].map(json.loads)
print(df["target"].value_counts())
print(len(df.iloc[0]["features"]))
df.head()

target
0.0    5120
1.0     335
Name: count, dtype: int64
7178


,clip,parent,id,features,target
0,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,"[0.0, 0.0, -3.6674314484295922, -3.86284365242...",1.0
1,Shoplifting001_x264_10,Shoplifting001_x264.mp4,2,"[0.0, 0.0, -4.92564021544842, -3.3099893501284...",1.0
2,Shoplifting001_x264_11,Shoplifting001_x264.mp4,1,"[0.0, 0.0, -5.068451145658348, -3.374883022677...",1.0
3,Shoplifting001_x264_11,Shoplifting001_x264.mp4,5,"[0.0, 0.0, -1.5864730109297471, -1.27144951956...",1.0
4,Shoplifting001_x264_12,Shoplifting001_x264.mp4,1,"[0.0, 0.0, -5.148482308764561, -3.334219974371...",1.0


In [ ]:
type(df.iloc[0]["features"]), type(df.iloc[0]["features"][0]), len(df.iloc[0]["features"])

(list, float, 7178)

# Augmentation

In [ ]:
import matplotlib.pyplot as plt

# # High-contrast color palette
# CONTRAST_COLORS = [
#     (0, 128, 255),   # Vivid Blue
#     (255, 128, 0),   # Deep Orange
#     (0, 255, 128),   # Bright Green
#     (255, 0, 128),   # Vivid Pink
#     (128, 0, 255),   # Strong Purple
#     (0, 255, 255),   # Cyan
#     (255, 255, 0),   # Yellow
# ]

# color_palette = sv.ColorPalette(CONTRAST_COLORS)
# Define high-contrast colors using hex values
HEX_COLORS = ['#3ab63f', '#459af5', '#ff5733', '#ffcc00', '#9900cc', '#00cccc', '#ff3399']
# joint_indices_required = {'Neck': 0, 'L_Eye': 1, 'R_Eye': 2, 'L_Shoulder': 3, 'R_Shoulder': 4, 'L_Elbow': 5, 'R_Elbow': 6, 'L_Wrist': 7, 'R_Wrist': 8, 'L_Hip': 9, 'R_Hip': 10}

COLORS = [
    "#FA2597",
    "#00FF00", "#FF1493", "#00FF00", "#FF1493",
    "#00FF00", "#FF1493", "#00FF00", "#FFD700",
    "#00BFFF", "#FFD700"
]

COLORS = [sv.Color.from_hex(color_hex=c) for c in COLORS]

# Create a color palette
color_palette = sv.ColorPalette.from_hex(HEX_COLORS)

def get_contrast_color(person_id):
    """Assigns a high-contrast color to a person ID."""
    return HEX_COLORS[person_id % len(HEX_COLORS)]  # Cycle through colors

def draw_annotated_frame(frame, bounding_box, keypoints):
    #draws a person onto a specific frame
    keypoints, _ = keypoints_to_list(keypoints)
    keypoints = convert_keypoints(keypoints, frame, to_pixel=True)
    keypoints = get_required(keypoints)
    labels = list(joint_indices_required.keys())
    frame_height, frame_width, _ = frame.shape
    person_boxes = []
    keypoint_list = []
    keypoint_labels = []

    # Convert bounding box from relative to absolute values
    x_center, y_center, width, height = bounding_box
    x_min = int((x_center - width / 2) * frame_width)
    y_min = int((y_center - height / 2) * frame_height)
    x_max = int((x_center + width / 2) * frame_width)
    y_max = int((y_center + height / 2) * frame_height)

    person_boxes.append([x_min, y_min, x_max, y_max])
    print(person_boxes)

    keypoint_list.append(keypoints)
    keypoint_labels.append(labels)

    # Convert to numpy array
    person_boxes = np.array(person_boxes)
    keypoints_np = np.array(keypoint_list)

    SKELETON = [
      [0, 1], [0, 2], [0, 3], [0, 4], [0, 9], [0, 10],
      [3, 5], [4, 6], [5, 7], [6, 8],
    ]
    # Create Supervision objects
    detections = sv.Detections(xyxy=person_boxes)
    keypoints = sv.KeyPoints(xy=keypoints_np)

    # Assign unique colors to each person_id
    box_annotator = sv.BoxAnnotator(
        color=sv.Color.BLUE,
        color_lookup = sv.ColorLookup.INDEX,
        thickness=1
    )

    edge_annotator = sv.EdgeAnnotator(color=sv.Color.GREEN, thickness=1, edges=SKELETON)
    vertex_label_annotator = sv.VertexLabelAnnotator(color=COLORS, text_color=sv.Color.BLACK, text_padding=1, border_radius=5)

    # Annotate the frame
    annotated_frame = box_annotator.annotate(scene=frame, detections=detections)
    annotated_frame = edge_annotator.annotate(scene=annotated_frame, key_points=keypoints)
    annotated_frame = vertex_label_annotator.annotate(scene=annotated_frame, key_points=keypoints) #, labels=keypoint_labels

    return annotated_frame


def plot_frame(frame):
  plt.figure(figsize=(10, 5))
  plt.imshow(frame)
  plt.show()


In [ ]:
import albumentations as A
import numpy as np
import cv2

def keypoints_to_list(keypoints, scale=True):
  keypoint_list = []
  labels = []
  for i in range(len(keypoints)):
    coords = [keypoints[i]["x"], keypoints[i]["y"]]
    keypoint_list.append(coords)
    labels.append(keypoints[i]["name"])
  return keypoint_list, labels

def convert_keypoints(keypoints, img, to_pixel=True):
  """converts a list of keypoint pairs from normalized coordinates to and from pixel coordinates"""
  height, width, _ = img.shape
  for keypoint in keypoints:
    keypoint[0] = keypoint[0] * width if to_pixel else keypoint[0] / width
    keypoint[1] = keypoint[1] * height if to_pixel else keypoint[1] / height
  return keypoints

def augment_frame(frame_path, person, transform, flip):
  #person parameter is the values of person at particular frame
  # cap = cv2.VideoCapture(clip_address)
  # cap.set(cv2.CAP_PROP_POS_FRAMES, frame_ind)
  # _, img = cap.read()
  # cap.release()
  img = cv2.imread(frame_path)
  keypoints, labels = keypoints_to_list(person["keypoints"])
  keypoints = convert_keypoints(keypoints, img, to_pixel=True)
  bounding_box = person["bbox"]

  pipeline = A.Compose([transform], bbox_params=A.BboxParams(format="yolo", clip=True, label_fields=["box_label"]), keypoint_params=A.KeypointParams(format="xy", remove_invisible=False))
  transformed_data = pipeline(image=img, keypoints=keypoints, bboxes=[bounding_box], box_label=[person["person_id"]])
  transformed_image = transformed_data["image"]
  height, width, _ = transformed_image.shape
  transformed_keypoints = transformed_data["keypoints"]
  transformed_box = transformed_data["bboxes"][0] if transformed_data["bboxes"] else [0] * 4

  #zero out invalid keypoints
  for i in range(len(transformed_keypoints)):
    if keypoints[i] == [0, 0] or transformed_keypoints[i][0] < 0 or transformed_keypoints[i][0] >= width or transformed_keypoints[i][1] < 0 or transformed_keypoints[i][1] >= height:
      transformed_keypoints[i] = [0, 0]

  #swap left and right keypoints if transform uses flip
  if flip:
    for i, joint in enumerate(labels) :
      if joint.startswith("L_"):
        transformed_keypoints[i], transformed_keypoints[i + 1] = transformed_keypoints[i + 1], transformed_keypoints[i]

  keypoints = convert_keypoints(keypoints, img, to_pixel=False)
  transformed_keypoints = convert_keypoints(transformed_keypoints, transformed_image, to_pixel=False)

  return transformed_box, transformed_keypoints, transformed_image

In [ ]:
from sklearn.model_selection import train_test_split
# load dataset
df = pd.read_csv(f"{output_path}/ids.csv")
counts = df["Has Required"].map(lambda s: len(json.loads(s)))
shoplifting = df["Label"] == 1
df = df[shoplifting]
print(f"dataframe shape: {df.shape}")

#TODO: Make it so only shoplifting samples in training set are augmented

dataframe shape: (340, 5)


In [ ]:
transforms = [A.HorizontalFlip(p=1.0), A.SafeRotate(p=1.0, limit=(-35, 35)), A.Compose([A.HorizontalFlip(p=1.0), A.SafeRotate(p=1.0, limit=(-35, 35))]), A.Perspective(p=1.0), A.Compose([A.HorizontalFlip(p=1.0), A.Perspective(p=1.0)])]
flip = [True, False, True, False, True] #indicates whether the ith transform uses horizontal flip

np.random.seed(42)
random.seed(42)

#df should only have shoplifting samples
for i in range(len(df)):
  row = df.iloc[i]
  parent = row["Parent Video"]
  clip = row["Clip"]
  id = row["ID"]
  # good_frames = json.loads(row["Has Required"])
  print(f"Processing {i + 1} of {len(df)}: {clip}: Person {id}")

  for t, transform in enumerate(transforms):
    # print(f"transform {i}")
    #loaded json should include the neck keypoints
    with open(f"{yolo_path}/{parent}/{clip}/persons/person_{id}/keypoints_0.json") as f:
      person = json.load(f)

    for frame in person:
      # if frame[frame.find('_')+1:] not in good_frames:
      #   continue
      frame_path = f"{yolo_path}/{parent}/{clip}/frames/{frame}.jpg"

      # print("Original")
      # plot_frame(draw_annotated_frame(cv2.imread(frame_path), person[frame][0]["bbox"], person[frame][0]["keypoints"]))

      transformed_box, transformed_keypoints, transformed_image = augment_frame(frame_path, person[frame][0], transform, flip[t])
      person[frame][0]["bbox"] = transformed_box
      for k in range(len(transformed_keypoints)):
        person[frame][0]["keypoints"][k]["x"] = transformed_keypoints[k][0]
        person[frame][0]["keypoints"][k]["y"] = transformed_keypoints[k][1]

      # print("Augmented")
      # plot_frame(draw_annotated_frame(transformed_image, person[frame][0]["bbox"], person[frame][0]["keypoints"]))

    with open(f"{yolo_path}/{parent}/{clip}/persons/person_{id}/keypoints_{t + 1}.json", 'w') as f:
      json.dump(person, f)

Processing 1 of 340: Shoplifting001_x264_10: Person 1
Processing 2 of 340: Shoplifting001_x264_10: Person 2
Processing 3 of 340: Shoplifting001_x264_11: Person 1
Processing 4 of 340: Shoplifting001_x264_11: Person 5
Processing 5 of 340: Shoplifting001_x264_12: Person 1
Processing 6 of 340: Shoplifting001_x264_12: Person 2
Processing 7 of 340: Shoplifting001_x264_13: Person 1
Processing 8 of 340: Shoplifting001_x264_13: Person 3
Processing 9 of 340: Shoplifting001_x264_14: Person 1
Processing 10 of 340: Shoplifting001_x264_14: Person 3
Processing 11 of 340: Shoplifting001_x264_15: Person 1
Processing 12 of 340: Shoplifting001_x264_15: Person 2
Processing 13 of 340: Shoplifting001_x264_16: Person 1
Processing 14 of 340: Shoplifting001_x264_16: Person 2
Processing 15 of 340: Shoplifting003_x264_62: Person 2
Processing 16 of 340: Shoplifting003_x264_63: Person 4
Processing 17 of 340: Shoplifting004_x264_38: Person 2
Processing 18 of 340: Shoplifting004_x264_38: Person 8
Processing 19 of 34

In [ ]:
#make csv for augmented dataset

ids_df = pd.read_csv(f"{output_path}/ids.csv")
indexes = pd.DataFrame({"Label": [0] + [1] * (len(transforms) + 1), "Index": [0] + list(range(len(transforms) + 1))})
augmented_df = ids_df.merge(indexes, on="Label")
drop_rows = []
for i in range(len(augmented_df)):
  row = augmented_df.iloc[i]
  clip = row["Clip"]
  parent = row["Parent Video"]
  id = row["ID"]
  index = row["Index"]
  print(f"Processing {i + 1} of {len(augmented_df)}: {clip}: Person {id} {index}")
  if not os.path.exists(f"{yolo_path}/{parent}/{clip}/persons/person_{id}/keypoints_{index}.json"):
    drop_rows.append(i)
    continue
augmented_df = augmented_df.drop(drop_rows)
print(len(augmented_df), augmented_df["Label"].value_counts())
augmented_df.to_csv(f"{output_path}/augmented_ids.csv", index=False)

Streaming output truncated to the last 5000 lines.
Processing 2263 of 7258: Shoplifting003_x264_82: Person 2 0
Processing 2264 of 7258: Shoplifting003_x264_83: Person 1 0
Processing 2265 of 7258: Shoplifting003_x264_83: Person 2 0
Processing 2266 of 7258: Shoplifting003_x264_84: Person 1 0
Processing 2267 of 7258: Shoplifting003_x264_86: Person 1 0
Processing 2268 of 7258: Shoplifting003_x264_86: Person 2 0
Processing 2269 of 7258: Shoplifting003_x264_87: Person 1 0
Processing 2270 of 7258: Shoplifting003_x264_87: Person 2 0
Processing 2271 of 7258: Shoplifting003_x264_87: Person 3 0
Processing 2272 of 7258: Shoplifting003_x264_88: Person 1 0
Processing 2273 of 7258: Shoplifting003_x264_89: Person 1 0
Processing 2274 of 7258: Shoplifting004_x264_0: Person 1 0
Processing 2275 of 7258: Shoplifting004_x264_0: Person 2 0
Processing 2276 of 7258: Shoplifting004_x264_0: Person 3 0
Processing 2277 of 7258: Shoplifting004_x264_1: Person 1 0
Processing 2278 of 7258: Shoplifting004_x264_1: Perso

In [ ]:
def feature_vector_from_row_augmented(row):
  person_path = f"{yolo_path}/{row['Parent Video']}/{row['Clip']}/persons/person_{row['ID']}/keypoints_{row['Index']}.json"
  good_frames = row["Has Required"]
  with open(person_path) as f:
    vector = get_feature_vector(f, good_frames, padding="after")
  return vector

def get_good_frames_augmented(row):
  print(f"processing {row['Clip']}; person id: {row['ID']}; keypoint index: {row['Index']}")
  person_path = f"{yolo_path}/{row['Parent Video']}/{row['Clip']}/persons/person_{row['ID']}/keypoints_{row['Index']}.json"
  with open(person_path) as f:
    person = json.load(f)
  add_missing(person)
  required = []
  for frame in person:
    keypoints = person[frame][0]["keypoints"]
    # if has_required(keypoints):
    if has_neck(keypoints):
      required.append(frame[frame.find('_')+1:])
  return required

augmented_df = pd.read_csv(f"{output_path}/augmented_ids.csv")
augmented_df["Has Required"] = augmented_df.apply(get_good_frames_augmented, axis=1)
counts = augmented_df["Has Required"].map(len)
augmented_df = augmented_df[counts > 0]

feature_df = pd.DataFrame()
feature_df["clip"] = augmented_df["Clip"]
feature_df["parent"] = augmented_df["Parent Video"]
feature_df["id"] = augmented_df["ID"]
feature_df["index"] = augmented_df["Index"]
feature_df["features"] = augmented_df.apply(feature_vector_from_row_augmented, axis=1)
feature_df["target"] = augmented_df["Label"]
print(len(feature_df), feature_df["target"].value_counts())
feature_df.to_csv(f"{output_path}/augmented_features_eyes.csv", index=False)
feature_df.head()

Streaming output truncated to the last 5000 lines.
processing Shoplifting003_x264_82; person id: 2; keypoint index: 0
processing Shoplifting003_x264_83; person id: 1; keypoint index: 0
processing Shoplifting003_x264_83; person id: 2; keypoint index: 0
processing Shoplifting003_x264_84; person id: 1; keypoint index: 0
processing Shoplifting003_x264_86; person id: 1; keypoint index: 0
processing Shoplifting003_x264_86; person id: 2; keypoint index: 0
processing Shoplifting003_x264_87; person id: 1; keypoint index: 0
processing Shoplifting003_x264_87; person id: 2; keypoint index: 0
processing Shoplifting003_x264_87; person id: 3; keypoint index: 0
processing Shoplifting003_x264_88; person id: 1; keypoint index: 0
processing Shoplifting003_x264_89; person id: 1; keypoint index: 0
processing Shoplifting004_x264_0; person id: 1; keypoint index: 0
processing Shoplifting004_x264_0; person id: 2; keypoint index: 0
processing Shoplifting004_x264_0; person id: 3; keypoint index: 0
processing Sho

,clip,parent,id,index,features,target
0,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,0,"[0.0, 0.0, -3.6674314484295922, -3.86284365242...",1.0
1,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,1,"[0.0, 0.0, -3.6674313386379156, -3.86284353678...",1.0
2,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,2,"[0.0, 0.0, -4.450594156446924, -4.687735715992...",1.0
3,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,3,"[0.0, 0.0, -4.353060628912291, -4.585005297432...",1.0
4,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,4,"[0.0, 0.0, -3.0521017632907252, -3.21472727947...",1.0


In [ ]:
df = pd.read_csv(f"{output_path}/augmented_features_eyes.csv")
df["features"] = df["features"].map(json.loads)
print(df["target"].value_counts())
print(len(df.iloc[0]["features"]))
df.head()

target
0.0    5120
1.0    2009
Name: count, dtype: int64
7178


,clip,parent,id,index,features,target
0,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,0,"[0.0, 0.0, -3.6674314484295922, -3.86284365242...",1.0
1,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,1,"[0.0, 0.0, -3.6674313386379156, -3.86284353678...",1.0
2,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,2,"[0.0, 0.0, -4.450594156446924, -4.687735715992...",1.0
3,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,3,"[0.0, 0.0, -4.353060628912291, -4.585005297432...",1.0
4,Shoplifting001_x264_10,Shoplifting001_x264.mp4,1,4,"[0.0, 0.0, -3.0521017632907252, -3.21472727947...",1.0


In [ ]:
# update neck keypoints


#Testing

In [ ]:
ids_df = pd.read_csv(f"{output_path}/ids.csv")
print("Original IDs", len(ids_df), ids_df["Label"].value_counts())
feature_df = pd.read_csv(f"{output_path}/features.csv")
print("Original features", feature_df["target"].value_counts())
idfs_df = pd.read_csv(f"{output_path}/ids_eyes.csv")
print("IDs with eyes", len(ids_df), ids_df["Label"].value_counts())
feature_df = pd.read_csv(f"{output_path}/features_eyes.csv")
print("Features with eyes", feature_df["target"].value_counts())
ids_df = pd.read_csv(f"{output_path}/augmented_ids.csv")
print("Augmented IDs with eyes", len(ids_df), ids_df["Label"].value_counts())
feature_df = pd.read_csv(f"{output_path}/augmented_features_eyes.csv")
print("Augmented features with eyes", feature_df["target"].value_counts())

Original IDs 5558 Label
0.0    5218
1.0     340
Name: count, dtype: int64
IDs with eyes 5558 Label
0.0    5218
1.0     340
Name: count, dtype: int64
Augmented IDs with eyes 6403 Label
0.0    5218
1.0    1185
Name: count, dtype: int64
Augmented features with eyes target
0.0    1103
1.0     492
Name: count, dtype: int64
Features with eyes target
0.0    2845
1.0     211
Name: count, dtype: int64
Original features target
0.0    2845
1.0     211
Name: count, dtype: int64


In [ ]:
features = ["features.csv", "features_eyes.csv", "augmented_features.csv", "augmented_features_eyes.csv"]
for feature in features:
  print(feature)
  df = pd.read_csv(f"{output_path}/{feature}")
  print(len(df), df["target"].value_counts())

features.csv
2246 target
0.0    2064
1.0     182
Name: count, dtype: int64
features_eyes.csv
1200 target
0.0    1103
1.0      97
Name: count, dtype: int64
augmented_features.csv
3073 target
0.0    2064
1.0    1009
Name: count, dtype: int64
augmented_features_eyes.csv
1634 target
0.0    1103
1.0     531
Name: count, dtype: int64
